In [1]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from matplotlib import pyplot as plt
import glob
import os

In [2]:
# df = pd.read_csv("examples.csv")
# for policy in df.policy_issue.unique():
#     df_policy = df[df.policy_issue == policy]
#     for stance in df_policy.stance.unique():
#         df_stance = df_policy[df_policy.stance == stance]
#         # print what is in favor and against policy
#         print(f"Policy: {policy}")
#         print(f"Stance: {stance}")
#         print("\n".join(df_stance["statement"].tolist()))
#         print()

In [3]:
qwen_results = pd.read_csv("out/2026-07-29T13:15:19.419179_Qwen-Qwen3.6-27B_by_policy.csv")
gpt_results = pd.read_csv("out/2026-07-27T01:19:51.551573_gpt-5.4-mini_by_policy.csv")

policies = [c[:-len("_pred")] for c in qwen_results.columns if c.endswith("_pred")]

def precision_recall_per_policy(df):
    rows = []
    for policy in policies:
        y_true = df[policy]
        y_pred = df[f"{policy}_pred"]
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        rows.append({
            "policy": policy,
            "precision": report["macro avg"]["precision"],
            "recall": report["macro avg"]["recall"],
        })
    return pd.DataFrame(rows).set_index("policy")

qwen_pr = precision_recall_per_policy(qwen_results).add_prefix("qwen_")
gpt_pr = precision_recall_per_policy(gpt_results).add_prefix("gpt_")

comparison = qwen_pr.join(gpt_pr)[["qwen_precision", "gpt_precision", "qwen_recall", "gpt_recall"]].round(3)
# add a row with the average precision and recall for each model
comparison.loc["average"] = comparison.mean()
comparison = comparison.sort_values(by="qwen_precision", ascending=False)
print(comparison.to_latex(formatters={"qwen_precision": "{:.2f}", "gpt_precision": "{:.2f}", "qwen_recall": "{:.2f}", "gpt_recall": "{:.2f}"}).replace("_", "-"))

\begin{tabular}{lrrrr}
\toprule
 & qwen-precision & gpt-precision & qwen-recall & gpt-recall \\
policy &  &  &  &  \\
\midrule
expanded-environ-protection & 0.85 & 0.79 & 0.92 & 0.85 \\
restrictive-migration-policy & 0.84 & 0.86 & 0.96 & 0.99 \\
restrictive-financial-policy & 0.77 & 0.72 & 0.90 & 0.81 \\
average & 0.77 & 0.78 & 0.88 & 0.84 \\
open-foreign-policy & 0.77 & 0.80 & 0.93 & 0.91 \\
expanded-social-welfare-state & 0.77 & 0.77 & 0.89 & 0.86 \\
liberal-society & 0.75 & 0.83 & 0.79 & 0.74 \\
liberal-economic-policy & 0.73 & 0.73 & 0.80 & 0.80 \\
law-and-order & 0.69 & 0.69 & 0.83 & 0.79 \\
\bottomrule
\end{tabular}



In [ ]:
ground_truth = pd.read_csv("annotations/stancepol.csv")

In [5]:
import re

def parse_run_model(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    ts_match = re.match(r"^(\d{4}-\d{2}-\d{2}T[\d:.]+)_(.+)$", stem)
    run_id, rest = ts_match.groups() if ts_match else (stem, stem)
    model = rest.split("_by_policy")[0]
    return run_id, model

files = glob.glob("out/*.csv")
annotations = pd.DataFrame()
for file in files:
    tmp = pd.read_csv(file)
    tmp["run"], tmp["model"] = parse_run_model(file)
    annotations = pd.concat([annotations, tmp], ignore_index=True)

policy_issues = ['expanded_environ_protection',
       'expanded_social_welfare_state', 'law_and_order',
       'restrictive_financial_policy', 'liberal_society',
       'open_foreign_policy', 'liberal_economic_policy',
       'restrictive_migration_policy']
print(len(annotations), "annotations loaded")
annotations[:2]

5000 annotations loaded


,id,text,close_categories,expanded_environ_protection,expanded_social_welfare_state,law_and_order,restrictive_financial_policy,liberal_society,open_foreign_policy,liberal_economic_policy,...,law_and_order_issue_relevant,law_and_order_stance_present,restrictive_migration_policy_issue_relevant,restrictive_migration_policy_stance_present,expanded_environ_protection_issue_relevant,expanded_environ_protection_stance_present,expanded_social_welfare_state_issue_relevant,expanded_social_welfare_state_stance_present,liberal_society_issue_relevant,liberal_society_stance_present
0,https://www.americanbazaaronline.com/2015/03/0...,Interview with noted immigration attorney Shee...,['restrictive_migration_policy'],neutral,neutral,neutral,neutral,neutral,neutral,neutral,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://actionnetwork.org/letters/tell-lawmake...,We're making progress in limiting the damage t...,"['liberal_economic_policy', 'restrictive_finan...",neutral,favor,neutral,favor,neutral,neutral,against,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
run_rows = []
for (run_id, model), group in annotations.groupby(["run", "model"]):
    row = {"run": run_id, "model": model}
    per_issue_f1 = []
    per_issue_kappa = []
    for issue in policy_issues:
        y_true = ground_truth[issue]
        y_pred = group[f"{issue}_pred"]

        n_missing = y_pred.isna().sum()
        if n_missing:
            print(f"{run_id}/{model}/{issue}: {n_missing} missing predictions treated as wrong")

        # fill missing predictions with a sentinel label so they count as
        # incorrect instead of crashing f1_score on a mix of str/NaN
        y_true = y_true.fillna("no_prediction").astype(str)
        y_pred = y_pred.fillna("no_prediction").astype(str)

        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        kappa = cohen_kappa_score(y_true, y_pred)

        row[f"{issue}_f1_macro"] = f1
        per_issue_f1.append(f1)
        per_issue_kappa.append(kappa)

    row["f1_macro_overall"] = np.mean(per_issue_f1)
    row["cohen_kappa_overall"] = np.mean(per_issue_kappa)
    run_rows.append(row)

run_results = pd.DataFrame(run_rows).sort_values("run").reset_index(drop=True)
print(run_results.to_markdown(index=False))

2026-07-27T00:37:37.980232/gpt-5.4-mini/restrictive_financial_policy: 1 missing predictions treated as wrong


2026-07-29T14:38:23.263666/google-gemma-4-31B-it/expanded_environ_protection: 200 missing predictions treated as wrong
2026-07-29T14:38:23.263666/google-gemma-4-31B-it/expanded_social_welfare_state: 200 missing predictions treated as wrong
2026-07-29T14:38:23.263666/google-gemma-4-31B-it/law_and_order: 200 missing predictions treated as wrong
2026-07-29T14:38:23.263666/google-gemma-4-31B-it/restrictive_financial_policy: 200 missing predictions treated as wrong
2026-07-29T14:38:23.263666/google-gemma-4-31B-it/liberal_society: 200 missing predictions treated as wrong
2026-07-29T14:38:23.263666/google-gemma-4-31B-it/open_foreign_policy: 200 missing predictions treated as wrong
2026-07-29T14:38:23.263666/google-gemma-4-31B-it/liberal_economic_policy: 200 missing predictions treated as wrong
2026-07-29T14:38:23.263666/google-gemma-4-31B-it/restrictive_migration_policy: 200 missing predictions treated as wrong
| run                        | model                     |   expanded_environ_prot

In [ ]:
results_cohen = []
annotator1 = pd.read_csv("annotations/dolma_tacl_sample_gold_cleaned_2.csv")
annotator2 = pd.read_csv("annotations/dolma_tacl_sample_gold_cleaned_1.csv")
maplabel = {"favor":"support", "against":"oppose", "neutral":"neutral"}
for policy in policy_issues:
    y_true = annotator1[policy].fillna("no_prediction").astype(str)
    y_pred = annotator2[policy].fillna("no_prediction").astype(str).map(maplabel)
    kappa = cohen_kappa_score(y_true, y_pred)
    results_cohen.append({"policy": policy, "cohen_kappa": kappa})
results_cohen = pd.DataFrame(results_cohen).sort_values("cohen_kappa", ascending=False).reset_index(drop=True)
# add row with average of all policies
avg_kappa = results_cohen["cohen_kappa"].mean()
results_cohen = pd.concat([results_cohen, pd.DataFrame([{"policy": "Average", "cohen_kappa": avg_kappa}])], ignore_index=True)
results_cohen["cohen_kappa"] = results_cohen["cohen_kappa"].round(3)
print(results_cohen.to_latex(float_format="%.3f", index=False))

\begin{tabular}{lr}
\toprule
policy & cohen_kappa \\
\midrule
expanded_environ_protection & 0.887 \\
restrictive_migration_policy & 0.801 \\
liberal_economic_policy & 0.696 \\
law_and_order & 0.658 \\
expanded_social_welfare_state & 0.623 \\
liberal_society & 0.552 \\
open_foreign_policy & 0.539 \\
restrictive_financial_policy & 0.483 \\
Average & 0.655 \\
\bottomrule
\end{tabular}



In [8]:
# print number of support, oppose, neutral for each policy in annotator1 and display it in a dataframe
results_counts = []
for policy in policy_issues:
    counts = annotator1[policy].value_counts(dropna=False).to_dict()
    results_counts.append({"policy": policy, "support": counts.get("support", 0), "oppose": counts.get("oppose", 0), "neutral": counts.get("neutral", 0)})
results_counts = pd.DataFrame(results_counts).sort_values("policy").reset_index(drop=True)
# print in latex and replace ("_" with " ")
results_counts_latex = results_counts.to_latex(index=False).replace("_", " ")
print(results_counts_latex)

\begin{tabular}{lrrr}
\toprule
policy & support & oppose & neutral \\
\midrule
expanded environ protection & 21 & 8 & 171 \\
expanded social welfare state & 27 & 15 & 158 \\
law and order & 14 & 15 & 171 \\
liberal economic policy & 40 & 28 & 132 \\
liberal society & 26 & 16 & 158 \\
open foreign policy & 21 & 19 & 160 \\
restrictive financial policy & 17 & 27 & 156 \\
restrictive migration policy & 11 & 15 & 174 \\
\bottomrule
\end{tabular}



In [9]:
# majority-class baseline: predict the most frequent label for every example
results_majority = []
for policy in policy_issues:
    y_true = ground_truth[policy].fillna("no_prediction").astype(str)
    majority_class = y_true.mode()[0]
    y_pred = pd.Series([majority_class] * len(y_true), index=y_true.index)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    results_majority.append({"policy": policy, "majority_class": majority_class, "f1_macro": f1})
results_majority = pd.DataFrame(results_majority).sort_values("policy").reset_index(drop=True)

avg_f1 = results_majority["f1_macro"].mean()
results_majority = pd.concat(
    [results_majority, pd.DataFrame([{"policy": "Average", "majority_class": "", "f1_macro": avg_f1}])],
    ignore_index=True,
)
results_majority["f1_macro"] = results_majority["f1_macro"].round(3)
print(results_majority.to_latex(index=False, float_format="%.3f").replace("_", " "))
results_majority

\begin{tabular}{llr}
\toprule
policy & majority class & f1 macro \\
\midrule
expanded environ protection & neutral & 0.307 \\
expanded social welfare state & neutral & 0.294 \\
law and order & neutral & 0.307 \\
liberal economic policy & neutral & 0.265 \\
liberal society & neutral & 0.294 \\
open foreign policy & neutral & 0.296 \\
restrictive financial policy & neutral & 0.292 \\
restrictive migration policy & neutral & 0.310 \\
Average &  & 0.296 \\
\bottomrule
\end{tabular}



,policy,majority_class,f1_macro
0,expanded_environ_protection,neutral,0.307
1,expanded_social_welfare_state,neutral,0.294
2,law_and_order,neutral,0.307
3,liberal_economic_policy,neutral,0.265
4,liberal_society,neutral,0.294
5,open_foreign_policy,neutral,0.296
6,restrictive_financial_policy,neutral,0.292
7,restrictive_migration_policy,neutral,0.310
8,Average,,0.296
